In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Check the unique languages and count how many rows each has
print("Languages found in the dataset:")
print(df['gh_lang'].value_counts())

Languages found in the dataset:
gh_lang
ruby    20000
Name: count, dtype: int64


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

file_path = '/content/drive/MyDrive/ML_CP/travis_torrent_final_2017.csv' # Update this if needed

print("🕵️‍♂️ Scanning 3GB dataset for Python logs in chunks... Please wait a minute...")

chunk_size = 100000
python_chunks = []
total_python_found = 0

# 1. READ IN CHUNKS (The Big Data Trick)
for chunk in pd.read_csv(file_path, chunksize=chunk_size, low_memory=False):
    # Filter the chunk for Python
    is_python = chunk[chunk['gh_lang'].astype(str).str.lower() == 'python']
    python_chunks.append(is_python)

    total_python_found += len(is_python)

    # Let's stop once we find 15,000 Python rows to save time
    if total_python_found >= 15000:
        print(f"✅ Reached {total_python_found} Python rows! Stopping scan.")
        break

# 2. COMBINE THE CHUNKS
df = pd.concat(python_chunks)
print(f"Total Python data loaded: {len(df)} rows")

# 3. TARGET PREPARATION (Passed vs Failed)
df_clean = df[df['tr_status'].isin(['passed', 'failed'])].copy()
y = df_clean['tr_status'].apply(lambda x: 1 if x == 'failed' else 0)

# 4. FEATURE SELECTION (Pre-Build Metrics)
features = [
    'gh_team_size',
    'git_diff_src_churn',
    'git_diff_test_churn',
    'gh_diff_files_modified',
    'gh_diff_files_added',
    'gh_sloc',
    'gh_is_pr',
    'gh_by_core_team_member'
]

X = df_clean[features].copy()

# 5. DATA CLEANING
X['gh_is_pr'] = X['gh_is_pr'].astype(int)
X['gh_by_core_team_member'] = X['gh_by_core_team_member'].astype(int)
X = X.fillna(0)

# 6. TRAIN / TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 7. TRAIN THE RANDOM FOREST
model = RandomForestClassifier(n_estimators=100, max_depth=10, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# 8. EVALUATE
predictions = model.predict(X_test)
accuracy = accuracy_score(y_test, predictions)

print(f"\n🚀 Python-Only ML Model Accuracy: {accuracy * 100:.2f}%\n")
print("Detailed Report (0 = Passed/Safe, 1 = Failed/High Risk):")
print(classification_report(y_test, predictions))

# 9. EXPORT
import joblib
joblib.dump(model, 'travis_python_risk_predictor.pkl')
print("\n✅ Final Model saved as 'travis_python_risk_predictor.pkl'!")

🕵️‍♂️ Scanning 3GB dataset for Python logs in chunks... Please wait a minute...
✅ Reached 23822 Python rows! Stopping scan.
Total Python data loaded: 23822 rows

🚀 Python-Only ML Model Accuracy: 81.96%

Detailed Report (0 = Passed/Safe, 1 = Failed/High Risk):
              precision    recall  f1-score   support

           0       0.84      0.90      0.87      3108
           1       0.78      0.66      0.72      1638

    accuracy                           0.82      4746
   macro avg       0.81      0.78      0.79      4746
weighted avg       0.82      0.82      0.82      4746


✅ Final Model saved as 'travis_python_risk_predictor.pkl'!
